# Render lai TOAN BO anh ECG khong co tieu de "ECG-ID" (Colab, tu chay, resume ben vung)

Ly do: tieu de `ECG-ID {id}` in tren anh la dinh danh phi lam sang — confound ma mo hinh co the hoc lam dac trung gia. `ecg_renderer.py` da bo tieu de; notebook nay render lai DUNG tap anh training v2 dang dung, ghi de tai cho.

**Cach dung:** Runtime > Run all. Cell 1 hien popup mount Drive — bam Allow. Phan con lai tu chay.

**Resume ben vung (quan trong):**
- Checkpoint ghi thang len **Drive** (`pulse_ptbxl_stage2/render_done.txt`) nen **song sot ca khi VM Colab bi thu hoi** — chay lai Run all se tiep tuc dung cho dang do.
- Ngoai ra tu do anh da render qua **thoi gian sua file (mtime)** de khoi phuc ca phan da lam o lan chay truoc (du checkpoint co mat).
- Bo qua file rac trung ten kieu `03440 (1).png`.
- Renderer NHANH (tai dung figure) co tu kiem chung pixel == renderer goc; neu khac thi tu dung renderer goc (an toan).

In [ ]:
# Cell 1 — mount Drive (bam Allow o popup)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Cell 2 — duong dan + clone repo (lay src/renderer) + env + cai dat
import os, subprocess, sys

ROOT       = '/content/drive/MyDrive/Universitat/LLAVA-med_GRADUATION_THESIS'
DATASET    = ROOT + '/Han/data/ptb-xl'                    # chua ptbxl_database.csv + records500/
OUT_IMAGES = ROOT + '/data/pulse_ptbxl_stage2/images'     # thu muc anh training v2 dung
REPO       = '/content/graduation_thesis'

if not os.path.exists(DATASET + '/ptbxl_database.csv'):
    hit = subprocess.run(['find','/content/drive/MyDrive','-name','ptbxl_database.csv','-not','-path','*/.*'],
                         capture_output=True, text=True, timeout=600).stdout.strip().splitlines()
    assert hit, 'khong thay ptbxl_database.csv — sua ROOT/DATASET o tren.'
    DATASET = os.path.dirname(hit[0]); print('auto DATASET =', DATASET)
if not os.path.isdir(OUT_IMAGES):
    hit = subprocess.run(['find','/content/drive/MyDrive','-type','d','-name','pulse_ptbxl_stage2'],
                         capture_output=True, text=True, timeout=600).stdout.strip().splitlines()
    assert hit, 'khong thay pulse_ptbxl_stage2 — sua OUT_IMAGES o tren.'
    OUT_IMAGES = hit[0] + '/images'; print('auto OUT_IMAGES =', OUT_IMAGES)

if not os.path.isdir(REPO):
    r = subprocess.run(['git','clone','--depth','1','--branch','research/pulse-v2',
                        'https://github.com/ewalliss/graduation_thesis.git', REPO],
                       capture_output=True, text=True, timeout=600)
    print('clone:', (r.stdout or '') + (r.stderr or ''))

os.environ['ECGLLAVA_PROJECT_ROOT'] = REPO
os.environ['ECGLLAVA_DATASET_ROOT'] = DATASET
os.environ['ECGLLAVA_OUTPUT_ROOT']  = ROOT + '/data/pulse_ptbxl_stage2'
if REPO not in sys.path:
    sys.path.insert(0, REPO)

get_ipython().system('pip -q install wfdb tqdm')

assert os.path.exists(DATASET + '/ptbxl_database.csv') and os.path.isdir(OUT_IMAGES)
print('DATASET   :', DATASET)
print('OUT_IMAGES:', OUT_IMAGES)

In [ ]:
# Cell 3 — imports, csv, renderer NHANH (tai dung figure) + tu kiem chung == renderer goc
import re, numpy as np, pandas as pd
from pathlib import Path
from PIL import Image
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt

import preprocessing_pipeline.preprocessing.image.ecg_renderer as R
from preprocessing_pipeline.preprocessing.signal.loader import load_wfdb_record_safe
from preprocessing_pipeline.preprocessing.signal.filter import apply_bandpass_filter

db = pd.read_csv(Path(DATASET)/'ptbxl_database.csv', index_col='ecg_id', skipinitialspace=True)
db.columns = db.columns.str.strip()

all_pngs = sorted(Path(OUT_IMAGES).glob('*.png'))
valid = [p for p in all_pngs if re.fullmatch(r'\d+', p.stem)]
junk  = [p for p in all_pngs if not re.fullmatch(r'\d+', p.stem)]
print(f'tong png={len(all_pngs)}  hop le={len(valid)}  bo qua(rac trung)={len(junk)}')

def build_fig():
    fig = plt.figure(figsize=(R.ECG_IMAGE_WIDTH_INCHES, R.ECG_IMAGE_HEIGHT_INCHES),
                     dpi=R.ECG_IMAGE_DPI, facecolor=R._PAPER_BG)
    gs = fig.add_gridspec(R._N_ROWS + 1, R._N_COLS, hspace=0.05, wspace=0.02,
                          top=0.93, bottom=0.04, left=0.02, right=0.99)
    leads = []
    for row in range(R._N_ROWS):
        for col in range(R._N_COLS):
            lead_idx = R._GRID_LEAD_INDICES[row][col]
            ax = fig.add_subplot(gs[row, col])
            R._draw_ecg_grid(ax, x_end=R._TIME_AXIS[-1], y_center=0.0, amplitude_mv=3.0)
            (ln,) = ax.plot(R._TIME_AXIS, np.zeros_like(R._TIME_AXIS),
                            color=R._SIGNAL_COLOR, linewidth=R._SIGNAL_LINEWIDTH, zorder=2)
            ax.text(0.01, 0.92, R.ECG_LEAD_NAMES[lead_idx], transform=ax.transAxes,
                    fontsize=6, fontweight='bold', color='#333333', verticalalignment='top')
            leads.append((ln, lead_idx, col))
    rax = fig.add_subplot(gs[R._N_ROWS, :])
    rt  = np.linspace(0, 10.0, R.SIGNAL_SAMPLING_RATE * 10)
    R._draw_ecg_grid(rax, x_end=10.0, y_center=0.0, amplitude_mv=3.0)
    (rln,) = rax.plot(rt, np.zeros_like(rt), color=R._SIGNAL_COLOR,
                      linewidth=R._SIGNAL_LINEWIDTH, zorder=2)
    rax.text(0.002, 0.92, 'II', transform=rax.transAxes, fontsize=6,
             fontweight='bold', color='#333333', verticalalignment='top')
    return fig, leads, rln

def fast_render(state, signal, out_path):
    fig, leads, rln = state
    seg = R._SAMPLES_PER_SEGMENT
    for ln, lead_idx, col in leads:
        ln.set_ydata(signal[lead_idx, col*seg:(col+1)*seg])
    rln.set_ydata(signal[1])
    fig.savefig(out_path, dpi=R.ECG_IMAGE_DPI)
    with Image.open(out_path) as img:
        if img.size != (R.ECG_IMAGE_OUTPUT_WIDTH_PX, R.ECG_IMAGE_OUTPUT_HEIGHT_PX):
            img.resize((R.ECG_IMAGE_OUTPUT_WIDTH_PX, R.ECG_IMAGE_OUTPUT_HEIGHT_PX),
                       Image.LANCZOS).save(out_path)

st = build_fig()
maxdiff = 0
for p in valid[200:205]:
    eid = int(p.stem); rec = str(db.loc[eid, 'filename_hr']).strip()
    sig = apply_bandpass_filter(load_wfdb_record_safe(rec))
    R.render_ecg_image(sig, Path('/content/_ref.png'), ecg_id=eid)
    fast_render(st, sig, Path('/content/_fast.png'))
    a = np.asarray(Image.open('/content/_ref.png').convert('RGB'), dtype=np.int16)
    b = np.asarray(Image.open('/content/_fast.png').convert('RGB'), dtype=np.int16)
    maxdiff = max(maxdiff, int(np.abs(a - b).max()))
plt.close(st[0])
USE_FAST = (maxdiff == 0)
print(f'pixel maxdiff fast-vs-goc = {maxdiff}  -> dung renderer {"NHANH" if USE_FAST else "GOC (an toan)"}')

In [ ]:
# Cell 4 — render lai TAT CA (song song, resume ben vung qua checkpoint Drive + mtime). Chay lai duoc bat ky luc nao.
import os, time, multiprocessing as mp
from pathlib import Path
from tqdm.auto import tqdm

POOL_N = 6
CKPT = os.path.dirname(OUT_IMAGES.rstrip('/')) + '/render_done.txt'   # tren DRIVE -> song sot khi VM recycle

# 1) resume tu checkpoint Drive (neu co)
done = set()
if os.path.exists(CKPT):
    done = {int(x) for x in open(CKPT).read().split() if x.strip().isdigit()}

# 2) seed them: anh da render no-ID o lan chay truoc (mtime < 2 ngay) — khoi phuc ca khi checkpoint mat
now = time.time()
recent = {int(p.stem) for p in valid if now - os.path.getmtime(p) < 2*86400}
print(f'checkpoint Drive={len(done)} | anh mtime moi (<2 ngay)={len(recent)}')
done |= recent

todo = [str(p) for p in valid if int(p.stem) not in done]
done_now = set(done)
with open(CKPT, 'w') as f:
    f.write('\n'.join(map(str, sorted(done_now))) + '\n')   # ghi lai checkpoint hop nhat len Drive
print(f'da xong (uoc tinh)={len(done)} | con lai={len(todo)} / {len(valid)}')

prog = {'done': len(done), 'ok': 0, 'fail': 0, 'total': len(valid)}
_W = {}
def _init():
    if USE_FAST:
        _W['state'] = build_fig()
def _render(path_str):
    eid = int(os.path.splitext(os.path.basename(path_str))[0])
    try:
        rec = str(db.loc[eid, 'filename_hr']).strip()
        sig = load_wfdb_record_safe(rec)
        if sig is None:
            return (eid, False)
        sig = apply_bandpass_filter(sig)
        if USE_FAST:
            fast_render(_W['state'], sig, Path(path_str))
        else:
            R.render_ecg_image(sig, Path(path_str), ecg_id=eid)
        return (eid, True)
    except Exception:
        return (eid, False)

def flush():
    with open(CKPT, 'w') as f:
        f.write('\n'.join(map(str, sorted(done_now))) + '\n')

if todo:
    with mp.Pool(POOL_N, initializer=_init) as pool:
        for eid, good in tqdm(pool.imap_unordered(_render, todo, chunksize=8), total=len(todo)):
            if good:
                prog['ok'] += 1; done_now.add(eid)
            else:
                prog['fail'] += 1
            prog['done'] += 1
            if prog['ok'] % 200 == 0:
                flush()   # ghi checkpoint len Drive dinh ky -> resume an toan
    flush()
print(f'XONG: ok={prog["ok"]} fail={prog["fail"]} | tong da xong={len(done_now)}/{prog["total"]}')
print('Neu bi dut giua chung: chi can Run all lai — se tu bo qua anh da xong va chay tiep.')

In [ ]:
# Cell 5 — kiem chung cuoi: mo 1 anh ngau nhien, xac nhan dinh anh TRONG (khong con ECG-ID)
from IPython.display import Image as IPyImage, display
import random
p = random.choice(valid)
print('anh:', p.name, '— dinh anh phai trong, khong co chu ECG-ID')
display(IPyImage(filename=str(p)))